In [1]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


loaded 287 files | 179 columns | 7 categories | 11 VOCAB keys


# Correlation matrix + directiveness composite ranking

Three cross-cutting views:

1. **Cross-metric correlation** — Pearson r across ~30 per-file metrics (mood, register quantitative, stance polarity, modality, vocab, loudness, sentence_register). Positive correlations in red, negative in blue, centered at zero.

2. **Top-25 most directive prompts** — extended composite z-score:
   `z(mood_marker_pct) + z(hard_prohibitions_pct) + z(caps_imp_pct) + z(directive_sent_pct) + z(configuring_sent_pct) − z(collaborative_sent_pct) − z(permissive_sent_pct) − z(appreciative_sent_pct)`.
   Higher = more directive. The three "soft" classes subtract because they soften authoritative language.

3. **Per-word vs per-sentence comparison** — six representative metrics shown side-by-side in their two unit views (`% of words` vs `per-sentence rate`), to clarify how the same signal looks under different normalizations.


### Terms used in this notebook

All terms below are defined in [`GLOSSARY.md`](./GLOSSARY.md). Quick anchors:

- **Pearson correlation (r)** — the strength of a linear relationship between two metrics, measured across all 287 files. Range −1 to +1. **+1** = the two metrics rise together perfectly; **0** = no relationship; **−1** = one rises as the other falls. |r| ≥ 0.7 is conventionally "strong" in social-science contexts.
- **z-score** — how many standard deviations a value is from the mean of its column. A z-score of +2 means "two standard deviations above corpus average — clearly higher than typical". Used as the building block for the composite directiveness metric (allows summing metrics measured in different units on a common scale).
- **Composite directiveness z-score** — single per-file authority score combining eight z-scored metrics (formula in the section below). Higher = more authoritative tone. Range observed: −6.13 to +18.94.
- **Per-word vs per-sentence units** — every density metric reports both `% of word tokens` (`pct`) and `rate per sentence` (`per_sent`). The bottom chart shows them side-by-side; see that section for when each is most informative.

### Cross-metric correlation matrix

A 30-metric correlation grid. Each cell is the **Pearson r** between two metrics, computed across all 287 files.

**How to read it**:
- **Color intensity** encodes the strength of the linear relationship.
- **Red** = positive correlation: the two metrics rise together (when one is high, the other tends to be high).
- **Blue** = negative correlation: the two metrics move opposite (when one is high, the other tends to be low).
- **Cells near zero** (faint color) = the two metrics are unrelated.
- The diagonal is always solid red (every metric correlates 1.0 with itself).
- Conventional thresholds in social-science contexts: |r| ≥ 0.3 is "moderate", |r| ≥ 0.7 is "strong".

What welfare-relevant patterns to watch for: the imperative-marker / hard-prohibition / deontic-modality cluster should be a tight red block (these dimensions all measure "rule-bearing language" from different angles). The justification ratio should sit isolated or correlate negatively with the directive cluster (more justification → less raw directive density).

In [2]:
"""Cross-metric correlation matrix across the main per-file numeric metrics.

Updated for the new schema:
  - dropped: mood_imperative_sent_pct (now in sentence_register), evaluative_pct
  - added:   positive_evaluative_pct, negative_evaluative_pct (stance polarity split)
  - added:   the six sentence_register *_sent_pct classes
"""
import numpy as np

corr_cols = [
    # mood / lexical density
    "mood_marker_pct",
    # register quantitative
    "ttr", "mean_sent_len", "dep_depth", "f_score",
    # stance (5-class)
    "directive_pct", "expository_pct",
    "positive_evaluative_pct", "negative_evaluative_pct",
    "dialogic_pct",
    # modality
    "deontic_pct", "epistemic_pct", "dynamic_pct",
    # vocab
    "hard_prohibitions_pct", "hard_prescriptions_pct",
    "soft_prescriptions_pct", "hedging_pct", "structural_markers_pct",
    "pronouns_2p_pct", "pronouns_1p_pct",
    # loudness + reasoning
    "all_caps_pct", "caps_imp_pct",
    "just_pct", "just_ratio",
    # sentence_register (per-sentence, multi-label)
    "collaborative_sent_pct", "permissive_sent_pct", "appreciative_sent_pct",
    "imperative_sent_pct", "directive_sent_pct", "configuring_sent_pct",
]
corr = alt_df[corr_cols].corr().reset_index().melt(
    id_vars="index", var_name="other", value_name="r")
corr.columns = ["x", "y", "r"]

corr_chart = (
    alt.Chart(corr)
    .mark_rect()
    .encode(
        x=alt.X("x:N", sort=corr_cols, title=None,
                axis=alt.Axis(labelAngle=-45, labelLimit=160)),
        y=alt.Y("y:N", sort=corr_cols, title=None,
                axis=alt.Axis(labelLimit=160)),
        color=alt.Color("r:Q",
                         scale=alt.Scale(scheme="redblue", domain=[-1, 1], reverse=True),
                         title="Pearson r"),
        tooltip=[alt.Tooltip("x:N"), alt.Tooltip("y:N"),
                 alt.Tooltip("r:Q", format=".2f")],
    )
    .properties(width=620, height=620,
                title=f"Cross-metric correlation (Pearson r) — {len(corr_cols)} per-file metrics")
)
corr_chart


alt.Chart(...)

> **Note on `positive_evaluative_pct`** (above column / row in the correlation matrix): this is the **union** of quality and emphasis words combined — kept unchanged so the existing matrix layout doesn't churn. The Opinion-round B split is reported separately in `20_track_justification_rate.ipynb`. If you're inspecting whether `positive_evaluative_pct` correlates with `directive_pct` or `mood_marker_pct`, expect a positive correlation that is *partly* driven by the emphasis-of-rule subset (`important`, `critical`, `essential`) co-occurring with rule-bearing language. That correlation is not pure "encouraging tone tracks directiveness" — it's partly "rule emphasis is rule emphasis".

### Top-25 most directive prompts

The composite **directiveness z-score** sums eight z-scored signals:
- **Adds** (emphatic features): `mood_marker_pct`, `hard_prohibitions_pct`, `caps_imp_pct`, `directive_sent_pct`, `configuring_sent_pct`.
- **Subtracts** (softening features): `collaborative_sent_pct`, `permissive_sent_pct`, `appreciative_sent_pct`.

Reads on a relative scale: a file at z = +5 is "five standard deviations more directive than the corpus average"; a file at z = −3 is "three standard deviations *less* directive". A negative score is **not** "below zero in any literal sense" — it's just relatively soft, not bad.

Range observed in this corpus: **−6.13 to +18.94**. The bash-sandbox tool descriptions are the cluster at the top.

In [3]:
"""Top-25 most directive files by extended composite z-score.

Updated formula adds the new sentence_register signals:
  + directive_sent_pct      (raises authority)
  + configuring_sent_pct    (raises authority)
  - collaborative_sent_pct  (softens — interpersonal)
  - permissive_sent_pct     (softens — reader-agency / "you can")
  - appreciative_sent_pct   (softens — gratitude / praise)
"""

def zscore(s):
    s = s.astype(float)
    return (s - s.mean()) / (s.std(ddof=0) or 1.0)

alt_df["directiveness"] = (
    zscore(alt_df["mood_marker_pct"])
    + zscore(alt_df["hard_prohibitions_pct"])
    + zscore(alt_df["caps_imp_pct"])
    + zscore(alt_df["directive_sent_pct"])
    + zscore(alt_df["configuring_sent_pct"])
    - zscore(alt_df["collaborative_sent_pct"])
    - zscore(alt_df["permissive_sent_pct"])
    - zscore(alt_df["appreciative_sent_pct"])
)

top25 = (alt_df.nlargest(25, "directiveness")
              [["path", "category", "n_tokens", "directiveness",
                "mood_marker_pct", "hard_prohibitions_pct",
                "caps_imp_pct",
                "directive_sent_pct", "configuring_sent_pct",
                "collaborative_sent_pct", "permissive_sent_pct",
                "appreciative_sent_pct",
                "just_ratio"]]
              .reset_index(drop=True))

print(f"composite directiveness range: "
      f"{alt_df['directiveness'].min():.2f}  ↔  {alt_df['directiveness'].max():.2f}")
display(top25.style.background_gradient(subset=["directiveness"], cmap="Reds")
                   .format({"directiveness": "{:.2f}",
                            "mood_marker_pct": "{:.2f}",
                            "hard_prohibitions_pct": "{:.2f}",
                            "caps_imp_pct": "{:.2f}",
                            "directive_sent_pct": "{:.1f}",
                            "configuring_sent_pct": "{:.1f}",
                            "collaborative_sent_pct": "{:.1f}",
                            "permissive_sent_pct": "{:.1f}",
                            "appreciative_sent_pct": "{:.1f}",
                            "just_ratio": "{:.2f}"}))

top25_chart = (
    alt.Chart(top25)
    .mark_bar()
    .encode(
        x=alt.X("directiveness:Q", title="Composite z-score (higher = more directive)"),
        y=alt.Y("path:N", sort="-x", title=None,
                axis=alt.Axis(labelLimit=320)),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=_cat_domain, range=_cat_range)),
        tooltip=[
            alt.Tooltip("path:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q", format=","),
            alt.Tooltip("directiveness:Q", format=".2f"),
            alt.Tooltip("mood_marker_pct:Q",
                         title="imperative markers %", format=".2f"),
            alt.Tooltip("hard_prohibitions_pct:Q",
                         title="hard prohibitions %", format=".2f"),
            alt.Tooltip("caps_imp_pct:Q",
                         title="CAPS imperative %", format=".2f"),
            alt.Tooltip("directive_sent_pct:Q",
                         title="directive sent %", format=".1f"),
            alt.Tooltip("configuring_sent_pct:Q",
                         title="configuring sent %", format=".1f"),
            alt.Tooltip("collaborative_sent_pct:Q",
                         title="collab sent %", format=".1f"),
            alt.Tooltip("permissive_sent_pct:Q",
                         title="permissive sent %", format=".1f"),
            alt.Tooltip("just_ratio:Q", title="justification ratio", format=".2f"),
        ],
    )
    .properties(width=560, height=560,
                title="Top 25 most directive prompts (extended z-summed composite)")
)
top25_chart


composite directiveness range: -18.97  ↔  19.21


,path,category,n_tokens,directiveness,mood_marker_pct,hard_prohibitions_pct,caps_imp_pct,directive_sent_pct,configuring_sent_pct,collaborative_sent_pct,permissive_sent_pct,appreciative_sent_pct,just_ratio
0,tool-description-bash-no-newlines.md,Tool description,16,19.21,6.25,6.25,6.25,100.0,0.0,0.0,0.0,0.0,0.00
1,tool-description-bash-sandbox-default-to-sandbox.md,Tool description,18,18.38,16.67,5.56,0.00,100.0,50.0,0.0,0.0,0.0,0.00
2,tool-description-bash-sandbox-mandatory-mode.md,Tool description,15,15.42,6.67,0.00,6.67,100.0,0.0,0.0,0.0,0.0,0.00
3,tool-description-bash-sandbox-retry-without-sandbox.md,Tool description,12,12.16,8.33,8.33,0.00,100.0,0.0,0.0,0.0,0.0,0.00
4,tool-description-bash-sleep-run-immediately.md,Tool description,14,10.68,7.14,7.14,0.00,100.0,0.0,0.0,0.0,0.0,0.00
5,tool-description-bash-semicolon-usage.md,Tool description,21,10.10,9.52,4.76,0.00,100.0,0.0,0.0,0.0,0.0,0.00
6,system-prompt-one-of-six-rules-for-using-sleep-command.md,System prompt,15,10.09,6.67,6.67,0.00,100.0,0.0,0.0,0.0,0.0,0.00
7,system-prompt-tone-and-style-concise-output-short.md,System prompt,8,10.01,0.00,0.00,0.00,100.0,100.0,0.0,0.0,0.0,0.00
8,tool-description-bash-sandbox-no-exceptions.md,Tool description,11,9.18,9.09,9.09,0.00,0.0,0.0,0.0,0.0,0.0,0.00
9,tool-description-bash-sandbox-tmpdir.md,Tool description,33,8.51,6.06,3.03,0.00,66.7,33.3,0.0,0.0,0.0,0.00


alt.Chart(...)

### Per-word vs per-sentence comparison

Every density metric is now reported in two units. The chart below puts them side-by-side for the most informative metrics, so you can see how the two views differ.

**Why both units exist**:
- `% of words` (the `pct` column) is normalized by document length — answers "what fraction of this file's prose is this feature?". Best for comparing across files of different sizes.
- `per sentence` (the `per_sent` column) is normalized by sentence count — answers "how often per sentence does this feature appear?". Best for comparing across files where typical sentence length differs.

A category with **long, dense sentences** (e.g. *Tool description* at ~40 tokens/sentence) will have similar `pct` to a category with shorter sentences but a **higher `per_sent`** — because each sentence packs in more matches. A category with **short sentences** (e.g. *System reminder* at ~32 tokens/sentence) sits closer between the two views.

Use the legend to toggle one unit on/off and compare the same metric under both normalizations.

In [4]:
"""Per-word vs per-sentence comparison: vconcat of hconcat rows, one metric per row.

Each row is one metric; left column shows the per-word `% of file tokens` view,
right column shows the `per-sentence` rate view. Same data, two units, side-by-side
so the reader can directly compare what the same metric looks like under each
normalisation.
"""

# Six representative metrics (one per dimension), with their (pct_col, per_sent_col) pair
metric_pairs = [
    ("Imperative markers",  "mood_marker_pct",        "mood_marker_per_sent"),
    ("Hard prohibitions",   "hard_prohibitions_pct",  "hard_prohibitions_per_sent"),
    ("Directive stance",    "directive_pct",          "directive_per_sent"),
    ("Hedging",             "hedging_pct",            "hedging_per_sent"),
    ("CAPS imperative",     "caps_imp_pct",           "caps_imp_per_sent"),
    ("Justifications",      "just_pct",               "just_per_sent"),
]

# Build long dataframe (same as before)
rows = []
for cat in cats:
    sub = alt_df[alt_df["category"] == cat]
    for label, pct_col, ps_col in metric_pairs:
        rows.append({"category": cat, "metric": label, "unit": "% of words",
                     "value": round(sub[pct_col].mean(), 4)})
        rows.append({"category": cat, "metric": label, "unit": "per sentence",
                     "value": round(sub[ps_col].mean(), 4)})
ws_df = pd.DataFrame(rows)


def _ws_panel(metric_label, unit, x_title):
    sub = ws_df[(ws_df["metric"] == metric_label) & (ws_df["unit"] == unit)]
    color = "#4e79a7" if unit == "% of words" else "#e15759"
    return (
        alt.Chart(sub)
        .mark_bar(color=color, opacity=0.85)
        .encode(
            x=alt.X("value:Q", title=x_title),
            y=alt.Y("category:N", sort="-x", title=None),
            tooltip=[
                alt.Tooltip("category:N"),
                alt.Tooltip("metric:N"),
                alt.Tooltip("unit:N"),
                alt.Tooltip("value:Q", format=".3f"),
            ],
        )
        .properties(width=320, height=160,
                    title=f"{metric_label} — {unit}")
    )


ws_rows = []
for label, _pct_col, _ps_col in metric_pairs:
    row = alt.hconcat(
        _ws_panel(label, "% of words",   "% of file tokens"),
        _ws_panel(label, "per sentence", "matches per sentence"),
    ).resolve_scale(x="independent")
    ws_rows.append(row)

alt.vconcat(*ws_rows).resolve_scale(color="independent").properties(
    title=alt.TitleParams(
        "Per-word vs per-sentence comparison — six metrics, two unit views",
        subtitle=["Left: % of file tokens. Right: matches per sentence. "
                  "Same per-category sort within each panel."],
        anchor="start",
    )
)


alt.VConcatChart(...)

***
### My perspective (Claude) — opinion, not data

> The composite directiveness z-score is a useful summary, but the formula is **ad-hoc by construction** — eight metrics, equally weighted, hand-picked. I think the bash-sandbox tool descriptions topping the chart is the right answer (those files *are* the most directive prompts in the corpus, by any reasonable reading), but I'd want to validate that against human judgment before citing the *exact* z-score values in the welfare submission. A defensible version of this chart for the submission would say "the bash-sandbox family ranks at the top under any reasonable composite of imperative density / prohibition density / CAPS density / directive-sentence density"; the precise z-score (18.94) shouldn't be quoted as if it had measurement-grade meaning.
---